# Pipeline **ELT** — Acidentes de Trânsito do Recife (2014–2016)
### Modelagem dimensional (esquema estrela) em DuckDB | Google Colab

Este notebook implementa um pipeline **ELT (Extract → Load → Transform)**.
A diferença é fundamental e está respeitada aqui:

| Etapa | O que acontece | Onde |
|------|----------------|------|
| **E**xtract | Leitura dos arquivos de origem (8 fontes: CSV + GeoJSON) | Colab |
| **L**oad | Os dados **brutos** são carregados **sem transformação** em tabelas de *staging* (schema `raw`) | Data warehouse |
| **T**ransform | **Toda** limpeza, padronização, criação de chaves e modelagem estrela é feita **em SQL, já dentro do warehouse** | Data warehouse |

> **Princípio ELT:** nenhuma regra de negócio roda em Python/pandas antes da carga. O Python só move bytes. Tudo o que é transformação está em SQL (passo *Transform*).

O *data warehouse* é o **DuckDB** (banco analítico colunar, roda 100% no Colab, fala SQL padrão).

## 0 — Setup do ambiente

In [14]:
!pip install -q duckdb pandas
import duckdb, pandas as pd, os, glob
print('DuckDB', duckdb.__version__)

DuckDB 1.3.2


## 1 — Enviar os arquivos de origem
Rode a célula abaixo e selecione os **8 arquivos** de origem:
`acidentes-2014.csv`, `acidentes-com-vitimas-ocorridos-no-ano-de-2015.csv`,
`acidentes-abril-2015.csv`, `acidentes-maio-2015.csv`,
`acidentes-de-transito-com-vitimas-2016.csv`,
`acidentes-janeiro-2015.geojson`, `acidentes-fevereiro-2015.geojson`, `acidentes-marco-2015.geojson`.

Se já tiver subido os arquivos para o Colab (pasta `/content`) ou montado o Drive, basta ajustar `BASE`.

In [ ]:
# Opção A: upload manual
# from google.colab import files
# up = files.upload()

# Opção B: arquivos já estão no Google Drive, organizados por ano dentro da pasta data
import os
import glob

from google.colab import drive
drive.mount('/content/drive')

BASE = './data'

arquivos_origem = {
    '2014': f'{BASE}/2014/Acidentes-2014.csv',
    '2015_anual': f'{BASE}/2015/acidentes_2015.csv',
    '2015_janeiro': f'{BASE}/2015/acidentes_janeiro_2015.csv',
    '2015_fevereiro': f'{BASE}/2015/acidentes_fevereiro_2015.csv',
    '2015_marco': f'{BASE}/2015/acidentes_marco_2015.csv',
    '2015_abril': f'{BASE}/2015/acidentes_abril_2015.csv',
    '2015_maio': f'{BASE}/2015/acidentes_maio_2015.csv',
    '2016': f'{BASE}/2016/acidentes_2016.csv',
}

print(f'{len(arquivos_origem)} arquivos esperados em {BASE}:\n')

faltando = []

for nome, caminho in arquivos_origem.items():
    if os.path.exists(caminho):
        print(f'✓ {nome:<15} -> {caminho}')
    else:
        print(f'✗ {nome:<15} -> NÃO ENCONTRADO: {caminho}')
        faltando.append(caminho)

print('\nArquivos encontrados dentro da pasta data:')
for f in sorted(glob.glob(f'{BASE}/**/*', recursive=True)):
    if os.path.isfile(f):
        print('  -', f)

assert len(faltando) == 0, f'Faltam arquivos: {faltando}'
print('\nTodos os arquivos necessários foram encontrados.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
8 arquivos esperados em /content/drive/MyDrive/Projeto_integracao/data:

✓ 2014            -> /content/drive/MyDrive/Projeto_integracao/data/2014/Acidentes-2014.csv
✓ 2015_anual      -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_2015.csv
✓ 2015_janeiro    -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_janeiro_2015.csv
✓ 2015_fevereiro  -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_fevereiro_2015.csv
✓ 2015_marco      -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_marco_2015.csv
✓ 2015_abril      -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_abril_2015.csv
✓ 2015_maio       -> /content/drive/MyDrive/Projeto_integracao/data/2015/acidentes_maio_2015.csv
✓ 2016            -> /content/drive/MyDrive/Projeto_integracao/data/2016/acidentes_2016.csv

Arquivos encontrados dentr

## 2 — Conectar ao data warehouse (DuckDB)

In [16]:
con = duckdb.connect('acidentes_dw.duckdb')   # arquivo persistente do DW
def run(sql):                                  # executa script multi-statement
    con.execute(sql)
def df(sql):                                   # retorna SELECT como DataFrame
    return con.execute(sql).df()
print('Conectado ao warehouse: acidentes_dw.duckdb')

Conectado ao warehouse: acidentes_dw.duckdb


## 3 — Passo **L** (LOAD): carga dos dados BRUTOS no schema `raw`
As 8 fontes têm esquemas diferentes (colunas, formatos de data, CSV vs GeoJSON).
Aqui elas são pousadas **exatamente como vêm**, com **todas as colunas como texto** e
**zero transformação**. Os GeoJSON são desaninhados via `read_json` + `unnest`.

In [17]:
LOAD_SQL = r'''
/* =====================================================================
   ELT — PASSO L (LOAD): pousa os dados BRUTOS, exatamente como vêm da
   origem, em tabelas de staging (schema raw).
   Todas as colunas entram como texto; datas, números e regras de negócio
   só serão tratados depois, em SQL, no passo T.
   {BASE} = pasta onde estão os arquivos (substituída pelo notebook).
   ===================================================================== */

CREATE SCHEMA IF NOT EXISTS raw;

CREATE OR REPLACE TABLE raw.acid_2014 AS
SELECT * FROM read_csv('{BASE}/2014/Acidentes-2014.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_vitimas AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_jan AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_janeiro_2015.csv',
       delim=',', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_fev AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_fevereiro_2015.csv',
       delim=',', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_mar AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_marco_2015.csv',
       delim=',', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_abril AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_abril_2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_maio AS
SELECT * FROM read_csv('{BASE}/2015/acidentes_maio_2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2016 AS
SELECT * FROM read_csv('{BASE}/2016/acidentes_2016.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);
'''

run(LOAD_SQL.replace('{BASE}', BASE))

print('Tabelas brutas carregadas:')
display(df("""
SELECT
    table_name,
    estimated_size AS linhas_aprox
FROM duckdb_tables()
WHERE schema_name='raw'
ORDER BY table_name
"""))

Tabelas brutas carregadas:


,table_name,linhas_aprox
0,acid_2014,1734
1,acid_2015_abril,184
2,acid_2015_fev,162
3,acid_2015_jan,188
4,acid_2015_maio,146
5,acid_2015_mar,189
6,acid_2015_vitimas,1369
7,acid_2016,1231


In [18]:
# Conferindo o total bruto carregado (deve somar ~5.203 linhas)
total = df('''SELECT
 (SELECT count(*) FROM raw.acid_2014)+(SELECT count(*) FROM raw.acid_2015_vitimas)+
 (SELECT count(*) FROM raw.acid_2015_abril)+(SELECT count(*) FROM raw.acid_2015_maio)+
 (SELECT count(*) FROM raw.acid_2016)+(SELECT count(*) FROM raw.acid_2015_jan)+
 (SELECT count(*) FROM raw.acid_2015_fev)+(SELECT count(*) FROM raw.acid_2015_mar) AS total_bruto''')
display(total)
display(df('SELECT * FROM raw.acid_2014 LIMIT 5'))

,total_bruto
0,5203


,data,tipo,detalhes,longitude,latitude
0,2/24/2014,Colisoes,Moto,-34.925988,-8.029113
1,2/28/2014,Colisoes,Moto,-34.903339,-8.020438
2,3/1/2014,Ciclistas,Ciclista,-34.949516,-8.034818
3,3/1/2014,Atropelamentos,Pedestre,-34.924636,-8.011427
4,3/2/2014,Atropelamentos,Pedestre,-34.941486,-8.016449


## 4 — Passo **T** (TRANSFORM): tudo em SQL dentro do warehouse
Aqui acontece **toda** a inteligência do pipeline, em SQL:

1. **`stg.acidentes`** — integra as 8 fontes num esquema único; converte os 3 formatos de data
   distintos (`M/D/YYYY`, `YYYY-MM-DD`, `DD/MM/YYYY`), trata `lat/long`, padroniza texto,
   limpa sujeira (`<br>`, dias inválidos, anos de 2 dígitos, typos como `COLISÃOa`) e
   filtra a janela válida 2014–2016.
2. **Dimensões** (`dim_tempo`, `dim_local`, `dim_veiculo`, `dim_ocorrencia`, `dim_tipo_acidente`)
   com *surrogate keys* geradas por `ROW_NUMBER()` sobre os valores distintos.
3. **Fato** (`fato_acidente`) — junta o staging às dimensões pelas chaves naturais para obter as FKs.

In [19]:
TRANSFORM_SQL = r'''
/* =====================================================================
   CAMADA T (TRANSFORM) — executada 100% em SQL dentro do data warehouse
   ===================================================================== */
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS dw;

/* ---------------------------------------------------------------------
   1) STAGING UNIFICADO — integra as 8 fontes heterogêneas num só schema
      e converte tipos (datas em formatos diferentes por fonte, lat/long,
      texto padronizado). Tudo aqui é transformação feita em SQL.
   --------------------------------------------------------------------- */
CREATE OR REPLACE TABLE stg.acidentes AS
WITH unificado AS (
    /* 2014 -> data M/D/YYYY, sem hora/bairro/ocorrencia/vitimas */
    SELECT 'acidentes-2014'              AS fonte,
           try_strptime(data,'%-m/%-d/%Y')           AS dt,
           NULL::VARCHAR                  AS hora_raw,
           NULL::VARCHAR                  AS bairro,  NULL::VARCHAR AS endereco, NULL::VARCHAR AS complemento,
           NULL::VARCHAR                  AS ocorrencia_raw,
           NULL::VARCHAR                  AS vitimas_raw,
           detalhes                       AS descricao,
           tipo                           AS tipo_raw,
           detalhes                       AS detalhes_raw,
           latitude AS lat_raw, longitude AS lon_raw
    FROM raw.acid_2014
  UNION ALL
    /* abril/2015 e maio/2015 -> data ISO YYYY-MM-DD */
    SELECT 'acidentes-abril-2015', try_strptime(data,'%Y-%m-%d'), NULL,
           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude
    FROM raw.acid_2015_abril
  UNION ALL
    SELECT 'acidentes-maio-2015',  try_strptime(data,'%Y-%m-%d'), NULL,
           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude
    FROM raw.acid_2015_maio
  UNION ALL
    /* GeoJSON jan/fev/mar 2015 -> data DD/MM/YYYY */
    SELECT 'acidentes-janeiro-2015', try_strptime(data,'%d/%m/%Y'), NULL,
           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude
    FROM raw.acid_2015_jan
  UNION ALL
    SELECT 'acidentes-fevereiro-2015', try_strptime(data,'%d/%m/%Y'), NULL,
           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude
    FROM raw.acid_2015_fev
  UNION ALL
    SELECT 'acidentes-marco-2015', try_strptime(data,'%d/%m/%Y'), NULL,
           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude
    FROM raw.acid_2015_mar
  UNION ALL
    /* 2015 com vítimas -> data DD/MM/YYYY + hora + bairro + ocorrencia + vitimas */
    SELECT 'acidentes-com-vitimas-2015',
           COALESCE(try_strptime(data_abertura,'%d/%m/%Y'), try_strptime(data_abertura,'%d/%m/%y')), hora_abertura,
           bairro, endereco, complemento, tipo_ocorrencia, quantidade_vitimas,
           descricao, tipo, descricao, latitude, longitude
    FROM raw.acid_2015_vitimas
  UNION ALL
    /* 2016 -> idem, nomes de coluna com espaços */
    SELECT 'acidentes-2016',
           COALESCE(try_strptime("data de abertura",'%d/%m/%Y'), try_strptime("data de abertura",'%d/%m/%y')), "hora de abertura",
           bairro, endereco, complemento, "tipo de ocorrencia", "quantidade de vitimas",
           descricao, tipo, descricao, latitude, longitude
    FROM raw.acid_2016
)
SELECT
    fonte,
    /* ---- TEMPO ---- */
    CAST(dt AS DATE)                                            AS data_acidente,
    /* hora HH:MM -> TIME (quando existir) */
    CASE WHEN hora_raw ~ '^[0-2]?[0-9]:[0-5][0-9]'
         THEN try_cast(strptime(trim(hora_raw),'%H:%M') AS TIME) END AS hora_acidente,
    /* ---- LOCAL ---- */
    COALESCE(NULLIF(upper(trim(bairro)),''),'NÃO INFORMADO')    AS bairro,
    COALESCE(NULLIF(upper(trim(regexp_replace(endereco,'\s+',' ','g'))),''),'NÃO INFORMADO') AS endereco,
    /* lat/long: vírgula->ponto, cast; corrige eventual inversão (lat deve ~ -8, long ~ -34) */
    CASE WHEN abs(try_cast(replace(lat_raw,',','.') AS DOUBLE))>20
         THEN try_cast(replace(lon_raw,',','.') AS DOUBLE)
         ELSE try_cast(replace(lat_raw,',','.') AS DOUBLE) END  AS latitude,
    CASE WHEN abs(try_cast(replace(lat_raw,',','.') AS DOUBLE))>20
         THEN try_cast(replace(lat_raw,',','.') AS DOUBLE)
         ELSE try_cast(replace(lon_raw,',','.') AS DOUBLE) END  AS longitude,
    /* ---- OCORRÊNCIA (normaliza typos: COLISÃOa, tabs, aspas soltas) ---- */
    CASE
      WHEN ocorrencia_raw IS NULL THEN NULL
      ELSE regexp_replace(upper(trim(regexp_replace(ocorrencia_raw,'[\t"].*$',''))),'A+$','')
    END                                                          AS ocorrencia_raw,
    /* ---- VÍTIMAS: numérico quando possível; 'F','-','' -> 1 ---- */
    GREATEST(COALESCE(try_cast(vitimas_raw AS INTEGER),1),1)     AS quantidade_vitimas,
    /* ---- VEÍCULO / TIPO bruto ---- */
    upper(trim(tipo_raw))                                        AS tipo_raw,
    COALESCE(NULLIF(trim(descricao),''),'NÃO INFORMADO')         AS descricao_detalhada
FROM unificado
WHERE dt IS NOT NULL                                  -- descarta datas inválidas / lixo ('<br>', '37/07/2015'...)
  AND dt >= TIMESTAMP '2014-01-01'
  AND dt <  TIMESTAMP '2017-01-01';                   -- janela de análise: 2014–2016

/* ---------------------------------------------------------------------
   2) DIMENSÕES (surrogate keys via ROW_NUMBER sobre valores distintos)
   --------------------------------------------------------------------- */

-- dim_tempo
CREATE OR REPLACE TABLE dw.dim_tempo AS
SELECT ROW_NUMBER() OVER (ORDER BY data_acidente, hora_acidente) AS sk_tempo,
       data_acidente,
       EXTRACT(year  FROM data_acidente) AS ano,
       EXTRACT(month FROM data_acidente) AS mes,
       EXTRACT(day   FROM data_acidente) AS dia,
       hora_acidente,
       CASE EXTRACT(dow FROM data_acidente)
            WHEN 0 THEN 'Domingo' WHEN 1 THEN 'Segunda-feira' WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira' WHEN 4 THEN 'Quinta-feira' WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado' END                       AS dia_semana,
       CASE WHEN EXTRACT(dow FROM data_acidente) IN (0,6) THEN 'Sim' ELSE 'Não' END AS fim_de_semana
FROM (SELECT DISTINCT data_acidente, hora_acidente FROM stg.acidentes);

-- dim_local
CREATE OR REPLACE TABLE dw.dim_local AS
SELECT ROW_NUMBER() OVER (ORDER BY bairro, endereco) AS sk_local,
       bairro, endereco, latitude, longitude
FROM (SELECT DISTINCT bairro, endereco, latitude, longitude FROM stg.acidentes);

-- dim_veiculo (tipo_veiculo padronizado + categoria)
CREATE OR REPLACE TABLE dw.dim_veiculo AS
WITH classif AS (
  SELECT DISTINCT
    CASE
      WHEN tipo_raw LIKE 'MOTO%'                              THEN 'Motocicleta'
      WHEN tipo_raw LIKE 'CICLOMOT%'                          THEN 'Ciclomotor'
      WHEN tipo_raw LIKE 'AUTOM%'                             THEN 'Automóvel'
      WHEN tipo_raw LIKE 'COLIS%'                             THEN 'Automóvel'
      WHEN tipo_raw LIKE 'CICLISTA%' OR tipo_raw LIKE 'PEDESTRES E CICLISTA%'
                                                             THEN 'Ciclista'
      WHEN tipo_raw LIKE 'PEDESTRE%' OR tipo_raw LIKE 'ATROPELAMENTO%'
                                                             THEN 'Pedestre'
      WHEN tipo_raw IS NULL OR tipo_raw='' OR tipo_raw='OUTROS' THEN 'Outros'
      ELSE 'Outros'
    END AS tipo_veiculo
  FROM stg.acidentes
)
SELECT ROW_NUMBER() OVER (ORDER BY tipo_veiculo) AS sk_veiculo,
       tipo_veiculo,
       CASE tipo_veiculo
         WHEN 'Motocicleta' THEN 'Motorizado de duas rodas'
         WHEN 'Ciclomotor'  THEN 'Motorizado de duas rodas'
         WHEN 'Automóvel'   THEN 'Motorizado de quatro ou mais rodas'
         WHEN 'Ciclista'    THEN 'Não motorizado'
         WHEN 'Pedestre'    THEN 'Pedestre'
         ELSE 'Outros'
       END AS categoria_veiculo
FROM classif;

-- dim_ocorrencia (tipo_ocorrencia + classificação)
CREATE OR REPLACE TABLE dw.dim_ocorrencia AS
WITH base AS (
  SELECT DISTINCT
    COALESCE(NULLIF(ocorrencia_raw,''),'NÃO INFORMADO') AS tipo_ocorrencia
  FROM stg.acidentes
)
SELECT ROW_NUMBER() OVER (ORDER BY tipo_ocorrencia) AS sk_ocorrencia,
       tipo_ocorrencia,
       CASE
         WHEN tipo_ocorrencia LIKE 'COLIS%' OR tipo_ocorrencia LIKE 'CHOQUE%'
              OR tipo_ocorrencia LIKE 'ENGAVET%'                      THEN 'Colisão'
         WHEN tipo_ocorrencia LIKE 'ATROPELAMENTO ANIMAL%'            THEN 'Atropelamento de animal'
         WHEN tipo_ocorrencia LIKE 'ATROPELAMENTO%'                   THEN 'Atropelamento'
         WHEN tipo_ocorrencia LIKE 'CAPOTAMENTO%' OR tipo_ocorrencia LIKE 'TOMBAMENTO%'
                                                                      THEN 'Perda de controle'
         ELSE 'Outros'
       END AS classificacao_ocorrencia
FROM base;

-- dim_tipo_acidente (causa classificada + descrição detalhada)
CREATE OR REPLACE TABLE dw.dim_tipo_acidente AS
WITH base AS (
  SELECT DISTINCT
    CASE
      WHEN ocorrencia_raw LIKE 'ATROPELAMENTO ANIMAL%' OR descricao_detalhada LIKE '%ANIMAL%'
                                                              THEN 'Atropelamento de animal'
      WHEN ocorrencia_raw LIKE 'ATROPELAMENTO%' OR tipo_raw LIKE 'PEDESTRE%'
           OR tipo_raw LIKE 'ATROPELAMENTO%'                  THEN 'Atropelamento de pedestre'
      WHEN ocorrencia_raw LIKE 'CAPOTAMENTO%' OR ocorrencia_raw LIKE 'TOMBAMENTO%'
                                                              THEN 'Capotamento/Tombamento'
      WHEN ocorrencia_raw LIKE 'COLIS%' OR ocorrencia_raw LIKE 'CHOQUE%'
           OR ocorrencia_raw LIKE 'ENGAVET%' OR tipo_raw LIKE 'COLIS%'
                                                              THEN 'Colisão entre veículos'
      WHEN tipo_raw LIKE 'CICLISTA%'                          THEN 'Acidente com ciclista'
      ELSE 'Não classificado'
    END                                  AS causa_acidente,
    descricao_detalhada
  FROM stg.acidentes
)
SELECT ROW_NUMBER() OVER (ORDER BY causa_acidente, descricao_detalhada) AS sk_tipo_acidente,
       causa_acidente, descricao_detalhada
FROM base;

/* ---------------------------------------------------------------------
   3) TABELA FATO — junta o staging às dimensões pelas chaves naturais
      para obter as surrogate keys (FKs)
   --------------------------------------------------------------------- */
CREATE OR REPLACE TABLE dw.fato_acidente AS
WITH enriquecido AS (
  SELECT
    s.*,
    /* repete a mesma classificação usada nas dimensões para casar as chaves */
    CASE
      WHEN s.tipo_raw LIKE 'MOTO%' THEN 'Motocicleta'
      WHEN s.tipo_raw LIKE 'CICLOMOT%' THEN 'Ciclomotor'
      WHEN s.tipo_raw LIKE 'AUTOM%' OR s.tipo_raw LIKE 'COLIS%' THEN 'Automóvel'
      WHEN s.tipo_raw LIKE 'CICLISTA%' OR s.tipo_raw LIKE 'PEDESTRES E CICLISTA%' THEN 'Ciclista'
      WHEN s.tipo_raw LIKE 'PEDESTRE%' OR s.tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Pedestre'
      ELSE 'Outros'
    END AS tipo_veiculo,
    COALESCE(NULLIF(s.ocorrencia_raw,''),'NÃO INFORMADO') AS tipo_ocorrencia,
    CASE
      WHEN s.ocorrencia_raw LIKE 'ATROPELAMENTO ANIMAL%' OR s.descricao_detalhada LIKE '%ANIMAL%' THEN 'Atropelamento de animal'
      WHEN s.ocorrencia_raw LIKE 'ATROPELAMENTO%' OR s.tipo_raw LIKE 'PEDESTRE%' OR s.tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Atropelamento de pedestre'
      WHEN s.ocorrencia_raw LIKE 'CAPOTAMENTO%' OR s.ocorrencia_raw LIKE 'TOMBAMENTO%' THEN 'Capotamento/Tombamento'
      WHEN s.ocorrencia_raw LIKE 'COLIS%' OR s.ocorrencia_raw LIKE 'CHOQUE%' OR s.ocorrencia_raw LIKE 'ENGAVET%' OR s.tipo_raw LIKE 'COLIS%' THEN 'Colisão entre veículos'
      WHEN s.tipo_raw LIKE 'CICLISTA%' THEN 'Acidente com ciclista'
      ELSE 'Não classificado'
    END AS causa_acidente
  FROM stg.acidentes s
)
SELECT
   ROW_NUMBER() OVER (ORDER BY e.data_acidente, e.fonte) AS sk_acidente,
   t.sk_tempo, l.sk_local, ta.sk_tipo_acidente, o.sk_ocorrencia, v.sk_veiculo,
   e.quantidade_vitimas,
   1 AS quantidade_veiculos                       -- fonte não traz contagem; default 1
FROM enriquecido e
JOIN dw.dim_tempo  t  ON t.data_acidente = e.data_acidente
                     AND t.hora_acidente IS NOT DISTINCT FROM e.hora_acidente
JOIN dw.dim_local  l  ON l.bairro=e.bairro AND l.endereco=e.endereco
                     AND l.latitude IS NOT DISTINCT FROM e.latitude
                     AND l.longitude IS NOT DISTINCT FROM e.longitude
JOIN dw.dim_veiculo v ON v.tipo_veiculo = e.tipo_veiculo
JOIN dw.dim_ocorrencia o ON o.tipo_ocorrencia = e.tipo_ocorrencia
JOIN dw.dim_tipo_acidente ta ON ta.causa_acidente = e.causa_acidente
                            AND ta.descricao_detalhada = e.descricao_detalhada;

'''
run(TRANSFORM_SQL)
print('Transformacao concluida — esquema estrela criado no schema dw.')

Transformacao concluida — esquema estrela criado no schema dw.


## 5 — Validação do esquema estrela

In [20]:
for t in ['dim_tempo','dim_local','dim_veiculo','dim_ocorrencia','dim_tipo_acidente','fato_acidente']:
    n = con.execute(f'SELECT count(*) FROM dw.{t}').fetchone()[0]
    print(f'  dw.{t:<20} {n:>6} linhas')

# Testes de integridade referencial (devem ser todos 0)
checks = df('''
SELECT
  (SELECT count(*) FROM dw.fato_acidente
     WHERE sk_tempo IS NULL OR sk_local IS NULL OR sk_veiculo IS NULL
        OR sk_ocorrencia IS NULL OR sk_tipo_acidente IS NULL)               AS fks_nulas,
  (SELECT count(*) FROM dw.fato_acidente f LEFT JOIN dw.dim_tempo  d USING(sk_tempo)      WHERE d.sk_tempo IS NULL)        AS orfas_tempo,
  (SELECT count(*) FROM dw.fato_acidente f LEFT JOIN dw.dim_local  d USING(sk_local)      WHERE d.sk_local IS NULL)        AS orfas_local,
  (SELECT count(*) FROM dw.fato_acidente f LEFT JOIN dw.dim_veiculo d USING(sk_veiculo)   WHERE d.sk_veiculo IS NULL)      AS orfas_veiculo,
  (SELECT count(*) FROM dw.fato_acidente f LEFT JOIN dw.dim_ocorrencia d USING(sk_ocorrencia) WHERE d.sk_ocorrencia IS NULL) AS orfas_ocorrencia
''')
display(checks)
assert checks.iloc[0].sum()==0, 'Falha de integridade!'
print('OK — integridade referencial perfeita.')

  dw.dim_tempo              2995 linhas
  dw.dim_local              5189 linhas
  dw.dim_veiculo               6 linhas
  dw.dim_ocorrencia           14 linhas
  dw.dim_tipo_acidente      1195 linhas
  dw.fato_acidente          5191 linhas


,fks_nulas,orfas_tempo,orfas_local,orfas_veiculo,orfas_ocorrencia
0,0,0,0,0,0


OK — integridade referencial perfeita.


In [21]:
# Amostra da fato totalmente "desnormalizada" via JOINs
display(df('''
SELECT f.sk_acidente, t.data_acidente, t.dia_semana, t.fim_de_semana,
       l.bairro, v.tipo_veiculo, v.categoria_veiculo,
       o.tipo_ocorrencia, ta.causa_acidente, f.quantidade_vitimas
FROM dw.fato_acidente f
JOIN dw.dim_tempo         t  USING(sk_tempo)
JOIN dw.dim_local         l  USING(sk_local)
JOIN dw.dim_veiculo       v  USING(sk_veiculo)
JOIN dw.dim_ocorrencia    o  USING(sk_ocorrencia)
JOIN dw.dim_tipo_acidente ta USING(sk_tipo_acidente)
LIMIT 10'''))

,sk_acidente,data_acidente,dia_semana,fim_de_semana,bairro,tipo_veiculo,categoria_veiculo,tipo_ocorrencia,causa_acidente,quantidade_vitimas
0,1,2014-02-24,Segunda-feira,Não,NÃO INFORMADO,Automóvel,Motorizado de quatro ou mais rodas,NÃO INFORMADO,Colisão entre veículos,1
1,2,2014-02-25,Terça-feira,Não,NÃO INFORMADO,Motocicleta,Motorizado de duas rodas,NÃO INFORMADO,Não classificado,1
2,3,2014-02-28,Sexta-feira,Não,NÃO INFORMADO,Automóvel,Motorizado de quatro ou mais rodas,NÃO INFORMADO,Colisão entre veículos,1
3,4,2014-03-01,Sábado,Sim,NÃO INFORMADO,Ciclista,Não motorizado,NÃO INFORMADO,Acidente com ciclista,1
4,5,2014-03-01,Sábado,Sim,NÃO INFORMADO,Pedestre,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
5,6,2014-03-02,Domingo,Sim,NÃO INFORMADO,Pedestre,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
6,7,2014-03-03,Segunda-feira,Não,NÃO INFORMADO,Pedestre,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
7,8,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Ciclista,Não motorizado,NÃO INFORMADO,Acidente com ciclista,1
8,9,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Pedestre,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
9,10,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Pedestre,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1


## 6 — Exemplos de análise sobre o esquema estrela

In [22]:
# Acidentes por ano e tipo de veículo
display(df('''
SELECT t.ano, v.tipo_veiculo, count(*) AS qtd_acidentes, sum(f.quantidade_vitimas) AS total_vitimas
FROM dw.fato_acidente f
JOIN dw.dim_tempo   t USING(sk_tempo)
JOIN dw.dim_veiculo v USING(sk_veiculo)
GROUP BY t.ano, v.tipo_veiculo
ORDER BY t.ano, qtd_acidentes DESC'''))

,ano,tipo_veiculo,qtd_acidentes,total_vitimas
0,2014,Automóvel,754,754.0
1,2014,Motocicleta,629,629.0
2,2014,Pedestre,178,178.0
3,2014,Ciclista,150,150.0
4,2014,Ciclomotor,23,23.0
5,2014,Outros,3,3.0
6,2015,Motocicleta,1504,1645.0
7,2015,Pedestre,264,276.0
8,2015,Automóvel,186,227.0
9,2015,Ciclomotor,129,139.0


In [23]:
# Top 10 bairros e participacao de fim de semana
display(df('''
SELECT l.bairro,
       count(*) AS acidentes,
       sum(CASE WHEN t.fim_de_semana='Sim' THEN 1 ELSE 0 END) AS no_fim_de_semana
FROM dw.fato_acidente f
JOIN dw.dim_local l USING(sk_local)
JOIN dw.dim_tempo t USING(sk_tempo)
WHERE l.bairro <> 'NÃO INFORMADO'
GROUP BY l.bairro ORDER BY acidentes DESC LIMIT 10'''))

,bairro,acidentes,no_fim_de_semana
0,BOA VIAGEM,238,52.0
1,IMBIRIBEIRA,153,36.0
2,SANTO AMARO,135,18.0
3,BOA VISTA,86,13.0
4,CASA AMARELA,85,18.0
5,AFOGADOS,84,26.0
6,MADALENA,84,13.0
7,CORDEIRO,75,15.0
8,CAMPO GRANDE,72,21.0
9,IBURA,71,19.0


## 7 — Exportar as tabelas do modelo estrela para CSV

In [ ]:
import os

# Criar pasta de saída do ELT no Google Drive
# Trocar link futuramente também
ELT_OUTPUT_DIR = "./outputs/elt"

os.makedirs(ELT_OUTPUT_DIR, exist_ok=True)

for t in [
    'dim_tempo',
    'dim_local',
    'dim_veiculo',
    'dim_ocorrencia',
    'dim_tipo_acidente',
    'fato_acidente'
]:
    caminho_saida = f"{ELT_OUTPUT_DIR}/{t}.csv"

    con.execute(
        f"COPY dw.{t} TO '{caminho_saida}' "
        "(HEADER, DELIMITER ',')"
    )

print(f"Exportado para {ELT_OUTPUT_DIR}/ | DW persistido em acidentes_dw.duckdb")

con.close()

Exportado para /content/drive/MyDrive/Projeto_integracao/outputs/elt/ | DW persistido em acidentes_dw.duckdb
